In [1]:
#문제 9
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/air_quality.csv")

Q1 = df['CO2'].quantile(0.25)
Q3 = df['CO2'].quantile(0.75)
IQR = Q3 - Q1

upper = Q3 + 1.5 * IQR
lower = Q1 - 1.5 * IQR

outliers = df[(df['CO2'] < lower) | (df['CO2'] > upper)]

print(len(outliers))

304


In [3]:
#문제 10
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/drinks.csv")

continent = df.groupby("continent")['beer_servings'].mean()
top_continent = continent.idxmax()

cond = df['continent'] == top_continent
df = df[cond]
df = df.sort_values('beer_servings', ascending=False)

result = int(df.iloc[4, 1])
print(result)

313


In [5]:
#문제 11
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/tourist.csv")

df['방문객합계'] = df['관광'] + df['공무'] + df['사업'] + df['기타']
df['관광객비율'] = df['관광'] / df['방문객합계']

a = df.sort_values('관광객비율', ascending=False).iloc[1, 3]
b = df.sort_values('관광', ascending=False).iloc[1, 2]

print(a+b)

441


In [8]:
#문제 12
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/chem.csv")

scaler = MinMaxScaler()
df['co_scaled'] = scaler.fit_transform(df[['co']])
df['nmhc_scaled'] = scaler.fit_transform(df[['nmhc']])

co_std = df['co_scaled'].std()
nmhc_std = df['nmhc_scaled'].std()
print(co_std, nmhc_std)

std_diff = round(co_std - nmhc_std, 3)
print(std_diff)


0.2856516497116944 0.3030617020578397
-0.017


In [17]:
#문제 13 방법(1)
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/loan.csv")

df['총대출액'] = df['신용대출'] + df['담보대출']

grouped = df.groupby(['지역코드', '성별'])['총대출액'].sum().unstack()


grouped['차이'] = abs(grouped[1] - grouped[2])
result = grouped['차이'].idxmax()
print(result)

4100000278


In [19]:
#문제 13 방법(2)
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/loan.csv")

df['총대출액'] = df['신용대출'] + df['담보대출']

aggfunc='sum'
pivot_result = df.pivot_table(index='지역코드', columns='성별', values='총대출액', aggfunc=aggfunc)
pivot_result['차이'] = abs(pivot_result[1] - pivot_result[2])
result = pivot_result['차이'].idxmax()
print(result)

4100000278


In [21]:
#문제 14 (방법1)
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/crime.csv")

cond1 = df["구분"] == "발생건수"
cond2 = df["구분"] == "검거건수"
df1 = df[cond1].iloc[:, 2:]
df2 = df[cond2].iloc[:, 2:]

df1 = df1.reset_index(drop=True)
df2 = df2.reset_index(drop=True)
df3 = df2 / df1
listbox = df3.idxmax(axis=1)

result = 0
for index, item in enumerate(listbox):
     result = result + df2.loc[index, item]
print(result)

7799


In [23]:
#문제 14 (방법2)
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/crime.csv")

df_long = pd.melt(df, id_vars=["연도", "구분"], var_name="범죄유형", value_name="건수")
df_pivot = df_long.pivot_table(index=["연도", "범죄유형"], columns="구분", values="건수").reset_index()

df_pivot["검거율"] = df_pivot["검거건수"]/df_pivot["발생건수"]

listbox = df_pivot.groupby("연도")["검거율"].idxmax()
max_crime=df_pivot.loc[listbox]

result=max_crime["검거건수"].sum()

print(result)

7799.0


In [25]:
#문제 15 (방법 1)
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/hr.csv")
m = df["만족도"].mean()
df["만족도"] = df["만족도"].fillna(m)
gm = df.groupby(["부서", "성과등급"])["근속연수"].transform("mean")

gm = gm.astype(int)
df["근속연수"] = df["근속연수"].fillna(gm)

df["연봉_근속연수"] = df["연봉"] / df["근속연수"]
df_year = df.nlargest(3,'연봉_근속연수')

A = df_year.iloc[-1]["근속연수"]

df["연봉_만족도"] = df["연봉"] / df["만족도"]
df_like = df.nlargest(2,'연봉_만족도')

B = df_like.iloc[-1]["교육참가횟수"]

result = A + B
print(result)

7.0


In [27]:
#문제 15 (방법 2)
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/hr.csv")

m = df["만족도"].mean()
df["만족도"] = df["만족도"].fillna(m)

gm = df.groupby(["부서", "성과등급"])["근속연수"].transform("mean")
gm = gm.astype(int)
df["근속연수"] = df["근속연수"].fillna(gm)

df["연봉_근속연수"] = df["연봉"] / df["근속연수"]
df_year = df.sort_values(by="연봉_근속연수", ascending=False)
A = df_year.iloc[2]["근속연수"]


df["연봉_만족도"] = df["연봉"] / df["만족도"]
df_like = df.sort_values(by="연봉_만족도", ascending=False)
B = df_like.iloc[1]["교육참가횟수"]

result = A + B
print(result)

7.0
